In [31]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from src.utils.db import get_engine

engine = get_engine()


In [32]:
df = pd.read_sql("SELECT * FROM user_base", engine)
print("로드:", df.shape)

# group_id 결측 제거
df = df[df['group_id'].notna()].copy()
print("group_id 결측 제거 후:", df.shape)

# pending_chat 음수 보정
n_neg = (df['pending_chat'] < 0).sum()
df.loc[df['pending_chat'] < 0, 'pending_chat'] = 0
print(f"pending_chat 음수 {n_neg}건 → 0 보정")

로드: (677081, 11)
group_id 결측 제거 후: (677080, 11)
pending_chat 음수 5건 → 0 보정


In [33]:
df[df['report_count'] > 50][['id','gender','point','report_count','alarm_count','pending_chat','pending_votes','ban_status','created_at']]

,id,gender,point,report_count,alarm_count,pending_chat,pending_votes,ban_status,created_at
0,831962,F,2248,253,40878,5499,110,N,2023-03-29 05:18:56.162368
49,834358,F,600426,210,912,5712,28,N,2023-04-03 10:37:13.518521
692,838541,F,100404,178,21442,4105,28,N,2023-04-19 16:36:39.906965
62524,912951,M,3170,60,1,0,108,N,2023-05-06 12:43:27.806935
366309,1242094,F,287,55,0,0,273,N,2023-05-14 01:24:52.715666
548684,1443618,M,2702,57,1,0,45,N,2023-05-22 05:56:54.239799
671687,1577954,F,99212,71,21,1801,11,N,2023-07-09 18:07:04.204081


In [34]:
df.info()

<class 'pandas.DataFrame'>
Index: 677080 entries, 0 to 677080
Data columns (total 11 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   id             677080 non-null  int64         
 1   gender         677080 non-null  str           
 2   point          677080 non-null  int64         
 3   is_push_on     677080 non-null  int64         
 4   created_at     677080 non-null  datetime64[us]
 5   ban_status     677080 non-null  str           
 6   report_count   677080 non-null  int64         
 7   alarm_count    677080 non-null  int64         
 8   pending_chat   677080 non-null  int64         
 9   pending_votes  677080 non-null  int64         
 10  group_id       677080 non-null  float64       
dtypes: datetime64[us](1), float64(1), int64(7), str(2)
memory usage: 62.0 MB


In [35]:
df.isnull().sum()

id               0
gender           0
point            0
is_push_on       0
created_at       0
ban_status       0
report_count     0
alarm_count      0
pending_chat     0
pending_votes    0
group_id         0
dtype: int64

In [36]:
df.describe()

,id,point,is_push_on,created_at,report_count,alarm_count,pending_chat,pending_votes,group_id
count,6.770800e+05,6.770800e+05,677080.000000,677080,677080.000000,677080.000000,677080.000000,677080.000000,677080.000000
mean,1.212968e+06,3.039165e+03,0.843100,2023-05-15 22:45:18.708990,0.037291,0.946653,0.098477,84.631143,37023.089168
min,8.319620e+05,0.000000e+00,0.000000,2023-03-29 05:18:56.162368,0.000000,0.000000,0.000000,0.000000,1.000000
25%,1.028076e+06,4.000000e+02,1.000000,2023-05-09 01:35:31.151565,0.000000,0.000000,0.000000,2.000000,18488.000000
50%,1.211728e+06,9.650000e+02,1.000000,2023-05-13 10:50:57.454012,0.000000,1.000000,0.000000,29.000000,35614.000000
75%,1.397904e+06,2.183000e+03,1.000000,2023-05-20 07:49:37.098974,0.000000,1.000000,0.000000,122.000000,54534.000000
max,1.583733e+06,8.850000e+08,1.000000,2024-05-09 08:31:17.710824,253.000000,40878.000000,5712.000000,3352.000000,84546.000000
std,2.138958e+05,1.076024e+06,0.363707,NaN,0.588107,56.115065,11.087809,123.262928,21997.706464


In [37]:
(df['pending_chat'] < 0).sum()

np.int64(0)

In [38]:
df[df['pending_chat'] < 0]

,id,gender,point,is_push_on,created_at,ban_status,report_count,alarm_count,pending_chat,pending_votes,group_id


In [39]:
for col in ['point','report_count','alarm_count','pending_chat','pending_votes']:
    n = (df[col] < 0).sum()
    if n > 0:
        print(f"{col}: {n}건")

In [40]:
df.loc[df['pending_chat'] < 0, 'pending_chat'] = 0

In [41]:
print((df['pending_chat'] < 0).sum())
df['pending_chat'].describe()

0


count    677080.000000
mean          0.098477
std          11.087809
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max        5712.000000
Name: pending_chat, dtype: float64

In [42]:
print("최종 행수:", df.shape[0], "(기대 677,080)")
print("고유 id:", df['id'].nunique())
print()
print("결측:")
print(df.isnull().sum())
print()
print("음수:")
for col in ['point','report_count','alarm_count','pending_chat','pending_votes']:
    n = (df[col] < 0).sum()
    print(f"  {col}: {n}")
print()
print("point 100만 초과:", (df['point'] > 1_000_000).sum())

최종 행수: 677080 (기대 677,080)
고유 id: 677080

결측:
id               0
gender           0
point            0
is_push_on       0
created_at       0
ban_status       0
report_count     0
alarm_count      0
pending_chat     0
pending_votes    0
group_id         0
dtype: int64

음수:
  point: 0
  report_count: 0
  alarm_count: 0
  pending_chat: 0
  pending_votes: 0

point 100만 초과: 4


In [43]:
OUT = PROJECT_ROOT / "data" / "processed" / "accounts_user_preprocessed.csv"
df.to_csv(OUT, index=False, encoding="utf-8-sig")

import os
print(f"저장 완료: {os.path.getsize(OUT)/1024**2:.1f} MB")

저장 완료: 40.7 MB


In [44]:
chk = pd.read_csv(OUT)
print("행수:", len(chk))
print("id 범위:", chk['id'].min(), "~", chk['id'].max())
print("고유 id:", chk['id'].nunique())

행수: 677080
id 범위: 831962 ~ 1583733
고유 id: 677080
